Run Cells Sequentially
Each cell is numbered in the order they should be run in

In [2]:
#import statements
import pandas as pd
import os
import json
import re
from datetime import datetime, timedelta, timezone
import struct
import TELTools as TT
import pyodbc
from concurrent.futures import ThreadPoolExecutor
import sqlalchemy

c:\Users\214225\AppData\Local\Programs\Python\Python37\lib\site-packages\snowflake\connector\vendored\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.6) or chardet (5.0.0)/charset_normalizer (2.0.9) doesn't match a supported version!
  RequestsDependencyWarning)


In [3]:
# globals
batch_datetimestamp = datetime.now()
CURRENT_TIMESTAMP = str(datetime.now()).replace(" ","_").replace(":","_").replace(".","_")
tel_tools_config_path = 'C:\Source\Repos\EnterpriseCloud\Configuration\TelTools_CRM.json'
ENV = os.environ['SNOWFLAKE_ENV']

In [4]:
pw = TT.getAzureSecret("USBIUSER-CRM").value
print(pw)

USBI_USER_AZURE001


In [7]:
srv = 'telcrmsrv.database.windows.net'
db = 'WorkOrder'
un ='USBIUSER'
chunk_size = 10000

conn_str = f'mssql+pymssql://{un}:{pw}@{srv}/{db}'
eng = sqlalchemy.engine.create_engine(conn_str,isolation_level='READ UNCOMMITTED',connect_args={"login_timeout": 10,"timeout":30},echo=False)
conn = eng.connect().execution_options(max_buffer= 10000,stream_results=True)


In [9]:
# POC for tel tools
srv = 'telcrmsrv.database.windows.net'
db = 'WorkOrder'
un ='USBIUSER'
chunk_size = 10000

conn_str = f'mssql+pymssql://{un}:{pw}@{srv}/{db}?charset=utf8' # issues with charset
eng = sqlalchemy.engine.create_engine(conn_str)
conn = eng.connect().execution_options(stream_results=True,max_buffer=chunk_size)

In [ ]:
# conn = eng.connect().execution_options(stream_results=True,yield_per=100)
result = eng.connect().execution_options(stream_results=True,max_buffer=100).execute(sqlalchemy.text("SELECT [Id] FROM [dbo].[tel_sm_module_type]"))

df = pd.DataFrame(result.partitions(size=chunk_size))
print(len(df))
print(df.head(5))

In [1]:
# poc into dataframe for yield_per, needs to loop
srv = 'telcrmsrv.database.windows.net'
db = 'WorkOrder'
un ='USBIUSER'
pw = 'USBI_USER_AZURE001'

#build this in tt
conn_str = f'mssql+pymssql://{un}:{pw}@{srv}/{db}?driver=ODBC+Driver+18+for+SQL+Server'
conn = sqlalchemy.engine.create_engine(conn_str).connect().execution_options(max_buffer=100,stream_results=True)
# pass the engine

#loop in the execution
total_batch = 0
with conn:
    result = conn.execute(sqlalchemy.text("SELECT [Id] FROM [dbo].[tel_sm_module_type]"))

    for res in result.partitions(size=100):
        cnt = len(res)
        print(cnt)
        total_batch += cnt
        print(total_batch)
        df = pd.DataFrame(res)
    print(total_batch)

NameError: name 'sqlalchemy' is not defined

In [1]:
# tet tel tools sql connection 
import TELTools as TT
srv = 'tcp:telcrmsrv.database.windows.net'
db = 'WorkOrder'
un ='USBIUSER'
pw = 'USBI_USER_AZURE001'
chunk_size = 100

conn_message, conn = TT.getsqlconn(srv,db,un,pw,chunk_size)

#loop in the execution
total_batch = 0
with conn:
    result = conn.execute(sqlalchemy.text("SELECT [Id] FROM [dbo].[tel_sm_module_type]"))

    for res in result.partitions(size=100):
        cnt = len(res)
        print(cnt)
        total_batch += cnt
        print(total_batch)
        df = pd.DataFrame(res)
    print(total_batch)

c:\Users\214225\AppData\Local\Programs\Python\Python37\lib\site-packages\snowflake\connector\vendored\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.6) or chardet (5.0.0)/charset_normalizer (2.0.9) doesn't match a supported version!
  RequestsDependencyWarning)


In [ ]:
with engine.connect() as conn:
    result = conn.execution_options(yield_per=100).execute(text("select * from table"))

    for partition in result.partitions():
        # partition is an iterable that will be at most 100 items
        for row in partition:
            print(f"{row}")

In [ ]:
pd.read_sql('SELECT [Id] FROM [dbo].[tel_sm_module_type]',conn)

In [78]:
from sqlalchemy.sql import text
sql = '''
    SELECT [Id] FROM [dbo].[tel_sm_module_type]
'''
with engine.connect() as conn:
    query = conn.execute(text(sql))         
print(query)

In [ ]:
import pandas as pd
import sqlalchemy as sql
import pyodbc

server = 'telcrmsrv.database.windows.net'
driver = 'SQL+Server'
db = 'WorkOrder'
myQuery = "SELECT 1"
engine = sql.create_engine('mssql+pyodbc://USBIUSER:USBI_USER_AZURE001@{}/{}?driver={}'.format(server, db, driver))
#df = pd.read_sql_query(sql=myQuery,con=engine)
conn = engine.connect()
df = pd.read_sql_query(sql=myQuery,con=engine)

In [ ]:
query = "SELECT Id FROM tel_sm_module_type"
df = pd.read_sql(query,conn)
print(df)

In [4]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'ODBC Driver 17 for SQL Server', 'SnowflakeDSIIDriver', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'ODBC Driver 18 for SQL Server']


In [2]:
import pyodbc
# create connection
server = 'telcrmsrv.database.windows.net'
db = 'WorkOrder'
un = 'USBIUSER@telcrmsrv.database.windows.net'
pw = 'USBI_USER_AZURE001'
cnxn = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+db+';UID='+un+';PWD='+pw+';')

In [1]:
#direct from ms
import pyodbc 
# Some other example server values are
# server = 'localhost\sqlexpress' # for a named instance
# server = 'myserver,port' # to specify an alternate port
server = 'tcp:telcrmsrv.database.windows.net' 
database = 'WorkOrder' 
username = 'USBIUSER' 
password = 'USBI_USER_AZURE001' 
# ENCRYPT defaults to yes starting in ODBC Driver 18. It's good to always specify ENCRYPT=yes on the client side to avoid MITM attacks.
cnxn = pyodbc.connect('DRIVER={ODBC Driver 18 for SQL Server};SERVER='+server+';DATABASE='+database+';ENCRYPT=yes;UID='+username+';PWD='+ password)
cursor = cnxn.cursor()

In [1]:
import pymssql  
conn = pymssql.connect(server='telcrmsrv.database.windows.net', 
                       user='USBIUSER@telcrmsrv.database.windows.net', 
                       password='USBI_USER_AZURE001', 
                       database='WorkOrder')

In [3]:
import pyodbc
SQL_ATTR_CONNECTION_TIMEOUT = 10
login_timeout = 1
connection_timeout = 3

# create connection
server = 'telcrmsrv.database.windows.net'
db = 'WorkOrder'
un = 'USBIUSER'
pw = 'USBI_USER_AZURE001'
conn_str = 'DRIVER={ODBC Driver 18 for SQL Server};SERVER='+server+';DATABASE='+db+';ENCRYPT=yes;UID='+un+';PWD='+pw

connection = pyodbc.connect(conn_str, timeout=login_timeout)

In [ ]:
query = "SELECT [Id] FROM [dbo].[tel_sm_module_type]" 
df = pd.read_sql(query,cnxn)
print(df)

NameError: name 'cnxn' is not defined

In [42]:
# load config entries from TELTools.json
def getTELToolsConfig():
    with open(tel_tools_config_path,'r',encoding='utf8') as f:
        tel_tools_json = json.load(f,strict=False)
        config_json = tel_tools_json.get('DataSets')
        paths_json = tel_tools_json.get('Paths')
    return config_json, paths_json

def getTELToolsConnections(tel_tools_connectors_path):
    with open(tel_tools_connectors_path,'r',encoding='utf8') as f:
        tel_tools_json = json.load(f,strict=False)
        conn_json = tel_tools_json.get('CRM').get(ENV)
    return conn_json

def getLastRowData(file_path):
    if os.path.isfile(file_path):
        with open(file_path,'r',encoding='utf8') as f:
            last_value_json = json.load(f,strict=False)
            LastRowData = last_value_json.get('LastRowData')
        return LastRowData

def setLastRowData(file_path,df,LastRowData):
    success = False
    try:
        incremental_value = df['modifiedon'].max() or datetime.strptime("01/01/1950","%d/%m/%Y")
        if incremental_value > datetime.strptime(LastRowData, "%m/%d/%Y %H:%M:%S"):
            js_data = {"LastRowData":incremental_value}
            # write out incremental value
            with open(file_path,'w') as f:
                f.write(json.dumps(js_data))  
            message = f'getLastRowData(): updated incremental value for {file_path} to {incremental_value}'
        else:
            message = f'getLastRowData(): incremental value not updated for {file_path}, {incremental_value} not greater than {LastRowData}.'
        print(message)
        success = True
        return success, message   
    except Exception as e:
        message = f'getLastRowData(): failed to write back new incremental value for: {file_path}: {e}'
        print(message)
        return success , message

def getSourceData(tbl,query,cnxn):
    success = False
    message = None
    json_list = None
    df = None
    row_count = 0
    try:
        df = pd.read_sql_query(query,cnxn)
        row_count = len(df)
        json_list= df.apply(lambda x: json.loads(x.to_json()), axis=1).to_list()
        success = True
        message = f'Read success for {tbl}.'
        return success, json_list, df, row_count, message
    except Exception as e:
        message = f'Failure to read data from sql server: {e}'
        return success, json_list, df, row_count, message


In [5]:
def processThreads(vals):
    cnxn = vals[0]
    config_json = vals[1]
    file_uploader_config = vals[2]
    last_value_path = vals[3]
    tbl = vals[4]
    
    thread_messages = []
    thread_success = False

    # get values from teltools_crm file for the table
    tbl_json = config_json.get(tbl)
    parameters = tbl_json.get('Parameters')[0]
    query = parameters[0].get('Query')
    entry_name = tbl_json.get('RuntimeConfiguration').get('EntryName') 
    
    #get values from file uploader config
    target_db = file_uploader_config.get(entry_name).get('SNOWFLAKE_DATABASE_NAME').upper()
    target_schema = file_uploader_config.get(entry_name).get('SNOWFLAKE_SCHEMA_NAME').upper()
    target_table_name = file_uploader_config.get(entry_name).get('SNOWFLAKE_TABLE_NAME').upper()

    # get incremental value
    LastRowData = None
    if '<<LastRowData>>' in query:
        file_path = last_value_path + tbl + '.json'
        LastRowData = getLastRowData(file_path) or "01/01/1950"
    if LastRowData: 
        query = query.replace('<<LastRowData>>',LastRowData)
    print(query)
    thread_messages.append(query)
    
    file_name = tbl + '-' + CURRENT_TIMESTAMP + '.json'

    read_success, json_list, df, row_count, read_message = getSourceData(tbl,query,cnxn)
    print(read_message)
    thread_messages.append(read_message)

    if row_count > 0 and read_success:
        # write data to SF
        write_message = TT.putSnowflakeDataFrame(target_db,target_schema,target_table_name,json_list,file_name,batch_datetimestamp,row_count)
        print(write_message)
        thread_messages.append(write_message)

        # write incremental value
        if not 'FAIL_SIGNAL' in write_message and LastRowData and read_success:
            incremental_write_success, incremental_message = setLastRowData(file_path,df,LastRowData)
            thread_messages.append(incremental_message)
            if incremental_write_success:
                thread_success = True
    else:
        message = f'no rows for {tbl}'
        thread_success = True
        print(message)
        thread_messages.append(message)
    
    return thread_messages, thread_success, tbl
 

In [ ]:
# main processor

# initialize vars
failed_files = "None"
files_processed = "FileName,RowCount\n"
messages_main = []

# load config data
config_json, paths_json = getTELToolsConfig()
file_uploader_path = paths_json.get('file_uploader_path')
last_value_path = paths_json.get('last_value_path')
tel_tools_connectors_path = paths_json.get('tel_tools_connectors_path')
threads = int(paths_json.get('threads'))

file_uploader_config = TT.getFileUploaderConfig(file_uploader_path)

# get connection data
conn_json = getTELToolsConnections(tel_tools_connectors_path)
srv = conn_json.get('Server')
db = conn_json.get('Database')
un = conn_json.get('UserName')
pwk = conn_json.get('PasswordKey')
pw = TT.getAzureSecret(pwk).value

# get sql connection
print("attempting sql connection")
message, cnxn = TT.getsqlconn(srv,db,un,pw)
print(message)
messages_main.append(message)

# iterate through tables
if cnxn:
    cnt = 1
    for tbl in config_json:
        result = []
        vals = []
        vals.append([cnxn,config_json,file_uploader_config,last_value_path,tbl])
    
    with ThreadPoolExecutor(max_workers=threads) as exe:
        print(f'Threading max workers: {exe._max_workers}.')
        # Maps the method with a list of values.
        result = exe.map(processThreads,vals)
     
    # process thread results thread_messages, tbl, thread_success
    for r in result:
        print(r[0])
        success = r[1]
        tbl = r[2]
        if success:
            files_processed = files_processed + tbl
        else:
            failed_files = failed_files.replace("None","") + tbl + "|"
else:
    message = f'No connection made to {srv}, {db}'
    print(message)
    messages_main.append(message)

cnxn.close()


In [ ]:
df = pd.read_sql_query(parameters, cnxn)
print(df.head(10))


In [17]:
# fix dictionary for active flag
with open(tel_tools_config_path,'r',encoding='utf8') as f:
    tel_tools_json = json.load(f,strict=False)
    
for item in tel_tools_json['DataSets']:
    tel_tools_json['DataSets'][item]['active'] = 'true'

   

In [ ]:
# fix dictionary for incremental field 
import json
tel_tools_config_path = 'C:\Source\Repos\EnterpriseCloud\Configuration\TelTools_CRM.json'
with open(tel_tools_config_path,'r',encoding='utf8') as f:
    tel_tools_json = json.load(f,strict=False)
    
for item in tel_tools_json['DataSets']:

    print(item)
    parameters = tel_tools_json['DataSets'][item]['Parameters'][0][0]
    query = parameters.get('Query')
    print(query)
    query_split = query.split('WHERE')
    print(len(query_split))
    if len(query_split) == 2:
        incremental_field = ''
        incremental_field = query_split[1].replace(" >= '<<LastRowData>>'","")
        incremental_field = incremental_field.strip()
        print(incremental_field)

        tel_tools_json['DataSets'][item]['incremental_field'] = incremental_field

In [28]:
# write back dictionary
with open(tel_tools_config_path,'w') as f:
    f.write(json.dumps(tel_tools_json, indent= 2)) 

In [ ]:
# convert encoding on incremental files
import os, io, json
# path = 'F:\\Shares\\Files\\TokyoElectron\\Powershell\\RuntimeConfig'
# path = '\\\\TELUSDWAPPSIT\\F$\\Shares\\Files\\TokyoElectron\\Powershell\\RuntimeConfig'
path = '\\\\TELUSDWAPPPROD\\F$\\Shares\\Files\\TokyoElectron\\Powershell\\RuntimeConfig'

file_list = os.listdir(path)

for f in file_list:
    if f.startswith('CRM-') and f.endswith('.json'):
        print(f)
        fpath = path + "\\" + f
        with open(fpath,'rb') as text:
            new_text = text.read().decode('utf-16-le').encode('utf-8').decode('utf-8')
         
            print(new_text)
            # write back file
            with open(fpath,'w',encoding='utf-8') as fnew:
                fnew.write(new_text)


In [ ]:
# fix file uploader case mismatch
# path = 'F:\\Shares\\Configuration\\FileUploader_Crm.json'
path = 'C:\\Source\\Repos\\EnterpriseCloud\\Configuration\\FileUploader_Crm.json'

with open(path,'r',encoding='utf8') as f:
    file_uploader_json = json.load(f,strict=False)
    
file_str = json.dumps(file_uploader_json, indent=2).upper()
file_str = file_str.replace('FALSE','"False"')
print(file_str)
    

In [64]:
# write out file uploader 
# write back dictionary
with open(path,'w') as f:
    f.write(json.loads(json.dumps(file_str, indent= 2))) 

In [ ]:
datetime.strptime("05/01/2023 05:57:32", "%m-%d-%Y %H:%M:%S")

In [200]:
#set custom variables
kcard = '23699'
user_email = 'nicholas.x.colabianchi@us.tel.com'

project = 'crm'.lower()
project_c = project.capitalize()
project_u = project.upper()

server = 'telcrmr2.database.windows.net'
db = 'r2sales'
SourceUserName = "USBIUSER"
SourcePassword = "USBIUSER-CRM"

#remaining variables are auto set
lzSchema = project_u
odsSchema = project_u
pubSchema = "ODS_" + project_u

LZdb = "LZ_DEV"
ODSdb = "LZ_DEV"
overwriteDDLs = True

repo_path = "C:\\Sources\\Repos\\EnterpriseCloud\\"
files_path = repo_path + "Notebooks\\CRM_Sales\\"
#print (files_path)

#snow file paths
snowLZ_path = repo_path+"Snow\\K."+kcard+".1__"+project+"base_LZTables.sql" 
snowODS_path = repo_path+"Snow\\K."+kcard+".2__"+project+"base_ODSTables.sql"

# sqlserver crm service info
tablespath = files_path + "tables.csv"
colspath = files_path + "columns.csv"
ddl_path = files_path + "ddls.txt"

teltools_path = repo_path+'Configuration\\TELTools.config'
tt_test_path = repo_path+'PowerShell\\TELTools-TestRunner.ps1'

fileuploader_path = repo_path+"Configuration\\FileUploader_CRM.config"

typeDict = {'nvarchar':'VARCHAR(16777216)',
            'numeric':'NUMBER(38,0)',
            'datetime':'TIMESTAMP_LTZ(9)',
            'datetime2':'TIMESTAMP_LTZ(9)',
            'char':'VARCHAR(16777216)',
            'varchar':'VARCHAR(16777216)',
            'boolean':'BOOLEAN',
            'decimal':'NUMBER(38,10)',
            'float':'NUMBER(38,10)',
            'str':'VARCHAR(16777216)',
            'bit':'BOOLEAN',
            'uniqueidentifier':'VARCHAR(16777216)',
            'int':'NUMBER(38,0)'}

In [201]:
#utilities
def getFile(path, split=True):
    f = open(path,'r')
    if split:
      filein = f.readlines()
    else:
      filein = f.read()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def get_tt_config(server, db, lzSchema, table, columns):
  columns_str = '], ['.join(columns)
  config = """,
      \""""+lzSchema.upper()+'-'+table+"""\": {
        "CacheDataSet": "No",
        "AuthenticationType": "SQLServer",
        "Source": \""""+lzSchema+"""\",
        "Securables": {
          "UserName": \""""+SourceUserName+"""\",
          "PasswordName": \""""+SourcePassword+"""\"
        },
        "Parameters": [
          [
            {
              "Query": "SELECT ["""+columns_str+"""] FROM [dbo].["""+table+"""]"
            },
            {
              "ServerInstance": \""""+server+"""\"
            },
            {
              "Database": \""""+db+"""\"
            }
          ]
        ],
        "OutputFile": {
          "FileDirectoryExtension": \""""+lzSchema+"""\\\\",
          "FileName": \""""+table+"""\",
          "Format": "json",
          "JSonDepth": 1,
          "SplitRowScheme": "Standard",
          "SplitRowCount": 25000
        }
      }"""
  
  return config


In [202]:
# adding tables to teltools file

# read csv template
df_tables = pd.read_csv(tablespath)
tables = df_tables['TABLE_NAME']
df_columns = pd.read_csv(colspath)

new_configs = ''
test_runs = ''

for table in tables:
    table_info = df_columns[df_columns.TABLE_NAME == table]
    columns = list(table_info.COLUMN_NAME)
    new_configs += get_tt_config(server, db, lzSchema, table, columns)
    test_runs += "\nWrite-TELDataSet   -SourceName 'SQLStatements'      -DataSetName '"+lzSchema+'-'+table+"'   		-DataSetData $RetrievedData;"

txt = ''.join(getFile(teltools_path))
start_idx = txt.find('\n  "SQLStatements": {')
end_idx = start_idx + txt[start_idx:].find('\n  }')-6
new_txt = txt[:end_idx]+new_configs+txt[end_idx:]
writeFile(teltools_path, new_txt)

# adding tables to teltools-testrunner script file
with open(tt_test_path, 'a+') as file:
    file.write(test_runs)

In [ ]:
# create snow file for LZ tables from tables csv

df_tables = pd.read_csv(tablespath)
tables = df_tables['TABLE_NAME']
df_columns = pd.read_csv(colspath)

if overwriteDDLs:
    ddlAction = """create or replace TABLE """ + lzSchema + "." + tables
else:
    ddlAction = """create TABLE """ + lzSchema + "." + tables + """ if not exists"""
    
#generate DDL
df_tables['lz_ddl'] = ddlAction + """ (\
_FILENAME VARCHAR(16777216),\
_FILEDATE TIMESTAMP_NTZ(9),\
_UPLOADDATE TIMESTAMP_NTZ(9),\
JSONFIELD VARIANT\
)COMMENT='Model """ + lzSchema + """ """ + tables + """ file.';"""

#write out LZ database table ddl commands to snow file
outstr = ""
outstr = "USE DATABASE <<LZ>>;\n"
for idx, row in df_tables.iterrows():
    outstr += row['lz_ddl'] + """
"""
print(outstr)
writeFile(snowLZ_path,outstr)

In [204]:
# create snow file for ODS tables from columns csv

df_tables = pd.read_csv(tablespath)
tables = df_tables['TABLE_NAME']
df_columns = pd.read_csv(colspath)

chars_path = repo_path + "Notebooks\\Sf_characters.csv"
sf_characters = pd.read_csv(chars_path)

# read csv template
df_cols = pd.read_csv(colspath)

outstr = "USE DATABASE <<ODS>>;\n"

for table in tables:

    csvData = df_cols[df_cols['TABLE_NAME']==table].copy(deep=True).reset_index(drop=True)
    
    #loop through characters df and replace any illegal characters
    for idx,row in sf_characters.iterrows():
        csvData['COLUMN_NAME']  = csvData['COLUMN_NAME'].str.replace(row['OLD'],row['NEW'])
    
    csvData = csvData.replace({'DATA_TYPE':typeDict})
    columnList = ''.join([row["COLUMN_NAME"]+ " " + row["DATA_TYPE"] + ",\n" for idx,row in csvData.iterrows()])

    if overwriteDDLs:
        ddlAction = 'create or replace TABLE ' + odsSchema + '.' + table + ' (\n'
    else:
        ddlAction = 'create TABLE ' + odsSchema  + ' if not exists (\n'
    controlColumns = '_FILEDATE TIMESTAMP_NTZ(9),\n _FILENAME varchar(16777216),\n AUDIT_LOG_ID varchar(128),\n _ROW_HASH varchar(128)\n);\n\n'
    
    outstr +=  ddlAction + columnList + controlColumns

writeFile(snowODS_path,outstr)

In [ ]:
# create the fileuploader.config entries 
# (remove existing entries if you need to overwrite them in teh fileuploader.config file)

#read existing file
from this import d


configFileList = getFile(fileuploader_path)

# create new list, remove empty lines
newFileList = []
for l in configFileList: 
    if l != "":
        newFileList.append(l)
  
# remove last line
newFileList = newFileList[:-1]

# create a working copy of df_tables to build config
df_tables_config = df_tables.copy(deep=True)

#check to see if entry exists in file uploader config, if it does drop it from the df
for idx, row in df_tables_config.iterrows():
    dfTableName = row['TABLE_NAME']
    config_exists = False
    for tn in newFileList:
        if dfTableName in tn:
            config_exists = True
    if config_exists:
        print('Table already in file uploader config, dropping',dfTableName,idx)        
        df_tables_config.drop(idx, inplace=True)
            
#create the new config entries
confstr = ""

for l in newFileList:
    confstr += l

confstr = confstr.rstrip()
confstr = confstr.rstrip('\r\n')
confstr = confstr.rstrip()
for idx, row in df_tables_config.iterrows():
    confTableName = row['TABLE_NAME'] 
    confFileName = row['TABLE_NAME']
    confName = row['TABLE_NAME']
    confstr += """,
    \"""" + confName + """\": {
        "SNOWFLAKE_SCHEMA_NAME": \"""" + lzSchema + """\",
        "SNOWFLAKE_DATABASE_NAME": "<<LZ>>",
        "SNOWFLAKE_WAREHOUSE_NAME": "Import_File_WH",
        "SNOWFLAKE_ROLE_NAME": "<<SYSADMIN>>",
        "SNOWFLAKE_TABLE_NAME": \"""" + confTableName + """\",
        "SNOWFLAKE_FILE_FORMAT_NAME": "CONTROL.POWERSHELL_JSON",
        "SNOWFLAKE_STAGE_TYPE": "NAMED",
        "SNOWFLAKE_STAGE_NAME": \"""" + lzSchema + """_STG",
        "FILE": "F:\\\\Shares\\\\Files\\\\"""+lzSchema+"""\\\\""" + confFileName + """.*.json",
        "SPLIT_FILE": false
    }"""
    
# add final bracket
confstr += """
}"""

# write out new config file
writeFile(fileuploader_path,confstr)

In [206]:
#**************************** LZ to ODS dbt model Utility**********************
#helper functions for DBT model

def getJinjaConfig(strategy,view,unique_keys):

	if unique_keys:
		jinjaConfig = """
{{ config(
	materialized='incremental',
	incremental_strategy='"""+ strategy +"""',
	alias='""" + view + """',
	unique_key = ['"""+ unique_keys +"""'],
	transient=false
) }} """

	else:
		jinjaConfig = """
{{ config(
	materialized='incremental',
	incremental_strategy='"""+ strategy +"""',
	alias='""" + view + """',
	transient=false
) }} """

	return jinjaConfig

#CREATE DBT MODEL
#(inputs are column list for source_cte,transform_cte and tablename )
def getDbtCode(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig,ifMerge):

	DBT_CODE = """

/************
Model file generated from """+project+"""DbtTemplateUtility.ipynb 
*************/

"""+ jconfig +"""

with source_data as (
	SELECT
		""" +col_s_string+ """,
		_FILEDATE,
		_FILENAME
	FROM """ + "{{ source('"+ odsSchema + "','" + tablename.upper() + "') }}" """
	"""+ ifMerge +"""
),

transform_data as (
	SELECT 
		""" +col_t_string+ """,
		_FILEDATE,
		_FILENAME,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
		SHA2('hash_me') AS _ROW_HASH -- Adds column to table on full refresh.
	FROM source_data
),

type_data as (
	SELECT
		"""+col_type_data_str+""",
		CAST(_FILEDATE as TIMESTAMP_NTZ(9)) as _FILEDATE,
		CAST(_FILENAME AS VARCHAR(16777216)) AS _FILENAME,
		CAST(AUDIT_LOG_ID AS VARCHAR(128)) AS AUDIT_LOG_ID,
		CAST(_ROW_HASH AS VARCHAR(128)) AS _ROW_HASH
	FROM transform_data
)

SELECT * FROM type_data
"""

	return DBT_CODE


#DBT CODE FOR INSERT-OVERWRITES
def getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig):

	DBT_CODE_I = """

/************
Model file generated from """+project+"""DbtTemplateUtility.ipynb 
*************/

"""+ jconfig +"""

with source_data as (
	SELECT
		""" +col_s_string+ """,
		_FILEDATE,
		_FILENAME
	FROM """ + "{{ source('"+ odsSchema + "','" + tablename.upper() + "') }}"  """
	WHERE _Filedate = (SELECT MAX(_FileDate) FROM """ + "{{ source('"+ odsSchema + "','" + tablename.upper() + "') }}"  """)
),

transform_data as (
	SELECT 
		""" +col_t_string+ """,
		_FILEDATE,
		_FILENAME,
		'{{ invocation_id }}' AS AUDIT_LOG_ID
	FROM source_data
),

type_data as (
	SELECT
		"""+col_type_data_str+""",
		CAST(_FILEDATE as TIMESTAMP_NTZ(9)) as _FILEDATE,
		CAST(_FILENAME AS VARCHAR(16777216)) AS _FILENAME,
		CAST(AUDIT_LOG_ID AS varchar(128)) AS AUDIT_LOG_ID
	FROM transform_data
)

SELECT * FROM type_data
"""
	
	return DBT_CODE_I


#CREATE SCHEMA FILE
def createSchema(df_tables_test,lzSchema,odsSchema,schemaPath):
        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + odsSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_LZ') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        df_tables_test['TABLE_NAME'] = df_tables_test['TABLE_NAME'].str.upper()

        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """"CRM_" + df_tables_test['TABLE_NAME'] + """\n    identifier: """"CRM_" + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        writeFile(path_write,final_schema)
		

In [ ]:
# extracts tables primary keys from ddl.txt file
ddls = getFile(ddl_path, split=False)

groups = re.findall('CREATE TABLE \[dbo\]\.\[(?:.|\n)*?GO', ddls)

pks = {}
working = []
for group in groups:
    table_name = re.search('CREATE TABLE \[dbo\]\.\[(.*)\]\(', group).groups()[0]
    pk_match = re.search('CONSTRAINT \[.*\] PRIMARY KEY *(?:CLUSTERED *)?\n*\(((?:.|\n)*?)\)', group)
    if pk_match:
        pks[table_name] = re.findall('\[(.*)\]', pk_match.groups()[0])

pks

In [ ]:
# Build lz_to_ods dbt models from view templates

tables = df_tables['TABLE_NAME']

# read csv template file and build list of required inputs
for table in tables:

    csvData = df_cols[df_cols['TABLE_NAME']==table].copy(deep=True).reset_index(drop=True).replace({'DATA_TYPE':typeDict})
    
    csvData["columnList"] = csvData["TABLE_NAME"] + " " + csvData["DATA_TYPE"] + ","
    
    #empty list for transform_cte,source_cte,type_data_cte columns 
    col_t = []
    col_s = []
    col_source = []
    col_type_data = []
    
    #type data columns
    for idx,row in csvData.iterrows():
        col_type_data.append('CAST("' + row["COLUMN_NAME"] + '" AS ' + row["DATA_TYPE"] + ') AS "' + row["COLUMN_NAME"].upper() + '"')
        col_t.append(row["COLUMN_NAME"])
        if row["DATA_TYPE"] == 'TIMESTAMP_LTZ(9)':
            col_s.append("REGEXP_REPLACE(JSONFIELD:\""+ row["COLUMN_NAME"] +"\", '\/Date\\\(|\\\)\/', '')::TIMESTAMP AS "+ row["COLUMN_NAME"])  
        else:
            col_s.append('JSONFIELD:"' + row["COLUMN_NAME"] + '"::STRING AS "' + row["COLUMN_NAME"].upper() + '"')  

    col_type_data_str = ', \n \t \t'.join(col_type_data)
    col_type_data_str = col_type_data_str.upper()

    # column string for transform cte
    col_t_string = '"' + '", \n \t \t"'.join(col_t) + '"'
    col_t_string = col_t_string.upper()
                 
    col_s_string = ', \n \t \t'.join(col_s)
    dbt_strategy = 'incremental'
    
    if table in pks:
        unique_fields = pks[table]    
        unique_fields_str = "','".join(unique_fields)    
        
        dbt_strategy = 'merge'
        
        unique_fields_qualify = ['JSONFIELD:"{0}"::STRING'.format(k) for k in unique_fields]
        unique_fields_qualify_str = ",".join(unique_fields_qualify)
        
        ifMerge = """{% if is_incremental() %}WHERE  _FILEDATE > IFNULL((SELECT MAX(_FILEDATE) FROM {{ this }}),'1970-01-01'){% endif %}
    QUALIFY (ROW_NUMBER() OVER (PARTITION BY """+unique_fields_qualify_str+"""  ORDER BY _FILENAME DESC,_FILEDATE DESC )) = 1"""
        
        jinjaConfig = getJinjaConfig(dbt_strategy,table,unique_fields_str)
        dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,table,odsSchema,jinjaConfig,ifMerge)

    else:
        dbt_strategy = 'insert_overwrite'
        unique_fields_str = ''
        
        jinjaConfig = getJinjaConfig(dbt_strategy,table,unique_fields_str)
        dbt_model = getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,table,odsSchema,jinjaConfig)

    #create dbt file
    dbt_path = repo_path + "dbt\\lz_to_ods\\models\\"+project+"\\"
    filename = table + ".sql"
    path_write = dbt_path + "CRM_" + filename.upper()

    writeFile(path_write,dbt_model)
    print('created file',filename)


In [209]:
#append lz_to_ods schema.yml file
dbt_path = repo_path + "dbt\\lz_to_ods\\models\\"+project+"\\"
tables = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path, split=False)
    for model in re.findall('\n      - name: (.*?)\n', schema_file):
        if model not in list(tables['TABLE_NAME']):
            existing_schemas.append(model)

#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
tables = pd.concat([tables,es],ignore_index=True)
createSchema(tables,lzSchema,odsSchema,dbt_path)

In [210]:
#append ods_to_pub schema.yml file
dbt_path = repo_path + "dbt\\ods_to_pub\\models\\"+project+"\\"
tables = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path, split=False)
    for model in re.findall('\n      - name: (.*?)\n', schema_file):
        if model not in list(tables['TABLE_NAME']):
            existing_schemas.append(model)

#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
tables = pd.concat([tables,es],ignore_index=True)
createSchema(tables,odsSchema,pubSchema,dbt_path)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt\\ods_to_pub\\models\\crm\\\\schema.yml'

In [ ]:
## create pub ODS_ views

%run ..\\Python\\dbtTools\\dbtGeneratePubViews.py "ODS_DEV" "$project_u" "C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models" "$user_email"


Arguments passed: DB: ODS_DEV  Schema: CRM Path: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models user: nicholas.x.colabianchi@us.tel.com
Schema.yml already exists, delete to refresh.
ODS_DEV ODS {{ source('ODS_CRM','SALES_ACCOUNT') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\crm\v_CRM_SALES_ACCOUNT.sql
ODS_DEV ODS {{ source('ODS_CRM','SALES_BUSINESSUNIT') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\crm\v_CRM_SALES_BUSINESSUNIT.sql
ODS_DEV ODS {{ source('ODS_CRM','SALES_CAMPAIGN') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\crm\v_CRM_SALES_CAMPAIGN.sql
ODS_DEV ODS {{ source('ODS_CRM','SALES_COM_COUNTRYMASTER') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\crm\v_CRM_SALES_COM_COUNTRYMASTER.sql
ODS_DEV ODS {{ source('ODS_CRM','SALES_COM_PRODUCTLINE') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\crm\v_CRM_SALES_COM_PRODUCTLINE.s